# Apache NiFi

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/nifi/nifi.ipynb)

Apache NiFi is a flow-based data distribution and processing system: you build a dataflow from processors that route, transform, enrich, and deliver FlowFiles, and NiFi handles the reliability, backpressure, provenance, and clustering around that flow.

## What NiFi is

+ **Flow-based programming**: processors are black boxes with typed relationships, and you compose behavior by wiring them together rather than writing a monolith.
+ **Guaranteed delivery**: FlowFiles sit in persisted repositories, so a restart or crash does not lose data that was already accepted.
+ **Backpressure and prioritization**: queues can tell upstream processors to stop, and you can prioritize which data moves first.
+ **Provenance**: every event on every FlowFile is recorded and queryable, down to the byte offset in a file.
+ **Zero-master clustering**: every node can do every job; coordinator and primary-node roles are elected, not statically assigned.

Typical interview framing: NiFi is strongest as a *data ingress, routing, and last-mile delivery* layer, especially when you need protocol adapters (SFTP, JMS, MQTT, REST), per-record enrichment, or audit-ready lineage without writing a service. It is not a replacement for a general compute engine.

A NiFi flow has five moving parts: processors that do the work, connections that hold FlowFiles in queues, process groups that nest flows, controller services that processors share, and ports that expose a group as an input or output. A scheduler runs processor invocations across a thread pool, and each invocation routes a FlowFile down one of the processor's relationships.

Worked topology for a partner file drop, wrapped in a process group with an input port and an output port:

```text
Input Port -> GetFile -> ConvertRecord -> RouteOnAttribute -> PutS3Object
                                                |
                                                +-> PutFile (dead letter)
Output Port (success only)
```

The group can be versioned and promoted as one unit, and a parent flow can feed it through the input port without knowing its internals.

Common mistake: building one giant flat canvas. A hundred processors connected on the root group cannot be versioned, tested, or reassigned as a unit, and every change is a change to production. Group related processors into a process group with an input and an output port, and version the group rather than the root.

## Core concepts

### FlowFiles and attributes

The critical design point is that a FlowFile is a **pointer**, not a blob. Content lives in the content repository; attributes live in memory and travel with the FlowFile reference.

| Attribute | Meaning |
|---|---|
| `uuid` | Unique FlowFile identifier, used in provenance |
| `filename` | Logical file name |
| `path` / `absolute.path` | Source directory |
| `mime.type` | Content type |
| `size` | Content size in bytes |
| `entryDate` | When the FlowFile entered NiFi |
| `lineageStartDate` | Start of the current lineage chain |
| `fragment.identifier`, `fragment.index` | Split/merge bookkeeping |

A FlowFile is an immutable record of an id, an attribute map, and a content claim. Attributes are strings held in the FlowFile repository and in heap; content is a claim reference into the content repository. A processor that changes content writes a new claim and repoints the FlowFile, and the old claim is released when its reference count reaches zero.

Worked attributes. After `UpdateAttribute`, a FlowFile can carry `filename = orders_20261004.json`, `path = /landing/orders/`, `mime.type = application/json`, and `size = 184320`. An expression such as `${filename:substringBefore('.')}` is evaluated against the attribute map without reading a byte of content, so routing a million FlowFiles by name costs almost no I/O.

Common mistake: treating attributes as unlimited. Every attribute is written into the FlowFile repository record and carried in memory, so a large attribute value or a million distinct attribute names inflates both the repository and heap. Keep routing and partitioning fields in attributes and leave bulky payload fields in content.

No. Attributes are metadata attached to the FlowFile reference and are never merged into the payload. A processor that needs the values to appear in the data must write them explicitly, for example with `UpdateRecord` or a `ConvertRecord` whose writer includes the field.

Common mistake: assuming downstream tools will see the attributes. Once a FlowFile is written to S3, HDFS, or Kafka, only the content survives unless a processor wrote the attribute into the record or the destination headers. If a consumer needs `ingest_date`, add it to the content with `UpdateRecord`.

The model below shows why a FlowFile is a pointer: attributes change freely, content lives once in the repository, and routing on attributes never reads the payload.

In [ ]:
import uuid
from dataclasses import dataclass, field


class ContentRepository:
    """Bytes live here once; FlowFiles hold reference-counted claims to them."""

    def __init__(self):
        self.claims, self.refcounts, self.reads = {}, {}, 0

    def write(self, data: bytes) -> str:
        claim = uuid.uuid4().hex
        self.claims[claim], self.refcounts[claim] = data, 1
        return claim

    def read(self, claim: str) -> bytes:
        self.reads += 1
        return self.claims[claim]

    def retain(self, claim):
        self.refcounts[claim] += 1

    def release(self, claim):
        self.refcounts[claim] -= 1
        if self.refcounts[claim] == 0:  # garbage once nothing points at it
            del self.claims[claim], self.refcounts[claim]


@dataclass(frozen=True)
class FlowFile:
    attributes: dict
    claim: str
    uuid: str = field(default_factory=lambda: uuid.uuid4().hex)


repo = ContentRepository()


def create(data, **attributes):
    return FlowFile(dict(attributes, size=str(len(data))), repo.write(data))


def clone(flowfile):  # e.g. a relationship connected to two downstream paths
    repo.retain(flowfile.claim)
    return FlowFile(dict(flowfile.attributes), flowfile.claim)


def update_attribute(flowfile, **changes):  # new FlowFile record, same content claim
    return FlowFile({**flowfile.attributes, **changes}, flowfile.claim, flowfile.uuid)


def replace_content(flowfile, transform):  # content change: new claim, old one released
    new_claim = repo.write(transform(repo.read(flowfile.claim)))
    repo.release(flowfile.claim)
    return FlowFile(dict(flowfile.attributes), new_claim, flowfile.uuid)


incoming = [create(b'{"order_id": %d}' % i, filename=f"orders_{i:04d}.json") for i in range(1_000)]
tagged = [update_attribute(ff, ingest_date="2026-10-04") for ff in incoming]
routed = [ff for ff in tagged if ff.attributes["filename"] < "orders_0500"]  # RouteOnAttribute
assert len(routed) == 500 and repo.reads == 0  # routing on metadata touched no content

archive_copy = clone(routed[0])
upper = replace_content(routed[0], bytes.upper)
assert archive_copy.claim in repo.claims and upper.claim != archive_copy.claim
assert repo.read(archive_copy.claim) == b'{"order_id": 0}'  # the clone still sees the original bytes
print("claims in repository:", len(repo.claims), "| content reads so far:", repo.reads)

### Repositories and large payloads

+ **FlowFile repository**: the write-ahead log of FlowFile metadata and pointers, so state survives restart.
+ **Content repository**: the actual bytes, stored in per-partition directories and referenced by claim.
+ **Provenance repository**: the event history used for lineage and replay.
+ **Database repository**: internal state for newer NiFi versions (component state, versions).

The content repository stores the bytes in per-partition directories. Small FlowFiles are appended into shared claim files so the repository does not hold one file per FlowFile, and each claim is reference counted. `nifi.content.claim.max.appendable.size` sets the largest FlowFile that may share a claim instead of getting its own file.

Worked sizing. At 8,000 FlowFiles/s with a 3 KB average, about 24.6 MB/s enters the flow. If the slowest branch retains ten minutes of data and you keep a 2x safety factor for claim overhead and archive cleanup, the repository needs roughly 24.6 MB/s x 600 s x 2, which is about 29.5 GB, before adding the flowfile and provenance repositories.

Common mistake: sizing for average payload and forgetting the archive. `nifi.content.repository.archive.max.usage.percentage` defaults to 50 percent, so a large share of the disk is held for archiving deleted content. Capacity-plan at the filesystem level and alert on content-repository disk usage, not only on the repository size property.

NiFi passes claim references, so moving a 50 GB FlowFile between processors does not copy 50 GB through the JVM. Streaming processors such as `PutS3Object` and `PutHDFS` read from the content repository straight to the destination, and `MergeContent` or `MergeRecord` combines many small FlowFiles without loading them all into heap.

Common mistake: a custom script or processor that calls `readAllBytes()`. It materializes the entire payload on heap, and a single large FlowFile can then cause a node-wide out-of-memory failure. Use the streaming API with bounded buffers in every custom processor and script.

### Processors and controller services

| Category | Examples |
|---|---|
| Ingest | `ConsumeKafka`, `GetFile`, `GetSFTP`, `ListenHTTP`, `ExecuteSQL`, `GetMongo` |
| Transform | `ConvertRecord`, `UpdateRecord`, `JoltTransformJSON`, `ReplaceText` |
| Enrich | `LookupRecord`, `InvokeHTTP`, `DistributedMapCacheClientService` |
| Route | `RouteOnAttribute`, `RouteOnContent`, `QueryRecord` |
| Split/merge | `SplitJson`, `SplitRecord`, `MergeContent`, `MergeRecord` |
| Deliver | `PutS3Object`, `PutHDFS`, `PublishKafka`, `PutDatabaseRecord`, `PutFile` |

A processor has properties, relationships, and scheduling. `schedulingStrategy` is TIMER_DRIVEN or EVENT_DRIVEN, `concurrentTasks` is how many invocations may run at once, and run duration is how long a task may hold its thread before yielding so it can gather more work into one batch.

Controller services are the shared objects a processor depends on: record readers and writers, DBCP connection pools, SSL contexts, distributed map caches, and schema registry clients. They are enabled once and referenced by property, so a pool of connections or one schema cache is reused across all tasks instead of being rebuilt per invocation.

Common mistake: raising `concurrentTasks` on a processor whose controller service cannot take the parallelism. A DBCP pool with `maxTotal = 10` behind a processor with `concurrentTasks = 20` spends its time waiting for a connection. Raise the service limit together with the processor concurrency, and watch the pool's active-connection metric.

### Relationships

A relationship is a named outcome. The processor declares which relationships it can produce, and every one must either be connected downstream or explicitly auto-terminated before NiFi will start it. The common names are `success`, `failure`, `retry`, `original`, `matched`, `unmatched`, `invalid`, and `not.found`.

```text
ValidateRecord --valid----> ConvertRecord --success--> PutS3Object
               --invalid--> PutFile (/dead-letter/orders)
               --failure--> RetryFlowFile --> ValidateRecord
```

Worked routing. `RouteOnAttribute` with `valid = ${amount:toNumber():ge(0)}` sends matching FlowFiles to `matched` and everything else to `unmatched`. Connecting `unmatched` to a dead-letter `PutFile` and auto-terminating `failure` on a sink are opposite decisions: the first preserves the record for review, the second deletes it.

Common mistake: auto-terminating `failure` on a sink processor to silence a warning. The FlowFile and its content are discarded permanently, and provenance shows a failure with no destination. Route failures to a dead-letter path, or at minimum through `LogAttribute` so the loss is recorded somewhere.

### Back pressure and prioritizers

Back pressure is per connection and has two independent thresholds: object count and data size. When either is reached, the upstream processor is simply not scheduled for that relationship, so FlowFiles stay in the queue and the pressure propagates one hop upstream until it reaches a source that can genuinely slow down.

+ **Back Pressure Object Threshold**: maximum number of FlowFiles in the queue (default 10,000).
+ **Back Pressure Data Size Threshold**: maximum bytes in the queue (default 1 GB).

Prioritizers decide which FlowFile leaves a queue when the downstream runs: `NewestFlowFileFirst`, `OldestFlowFileFirst`, `PriorityAttributePrioritizer`, and `FirstInFirstOut`. They do not increase throughput. Worked: a queue growing 500 FlowFiles/s reaches a 50,000-object threshold in 100 seconds, which is the recovery window the downstream gets before the source stalls.

Common mistake: raising the object threshold to a huge number to make back pressure warnings disappear. The queue then grows until the node exhausts heap or the flowfile repository fills, turning a graceful stall into an outage. Size the threshold to what the downstream can absorb and the node can hold.

In [ ]:
import heapq
import itertools


class Connection:
    """A queue with object-count and data-size back pressure plus a prioritizer."""

    def __init__(self, max_objects, max_bytes, prioritizer="oldest_first"):
        self.max_objects, self.max_bytes = max_objects, max_bytes
        self.prioritizer, self.heap, self.bytes = prioritizer, [], 0
        self.order = itertools.count()

    def is_full(self):  # either threshold stops the upstream processor from being scheduled
        return len(self.heap) >= self.max_objects or self.bytes >= self.max_bytes

    def offer(self, flowfile, entered_at):
        key = entered_at if self.prioritizer == "oldest_first" else -entered_at
        heapq.heappush(self.heap, (key, next(self.order), flowfile))
        self.bytes += int(flowfile.attributes["size"])

    def poll(self):
        _, _, flowfile = heapq.heappop(self.heap)
        self.bytes -= int(flowfile.attributes["size"])
        return flowfile


def simulate(prioritizer, ticks=20, produce_per_tick=50, consume_per_tick=30):
    queue = Connection(max_objects=200, max_bytes=10_000_000, prioritizer=prioritizer)
    produced = stalled_ticks = 0
    delivered = []
    for tick in range(ticks):
        if queue.is_full():
            stalled_ticks += 1  # producer yields; nothing is dropped
        else:
            for _ in range(produce_per_tick):
                queue.offer(create(b"x" * 100, seq=str(produced)), entered_at=produced)
                produced += 1
        for _ in range(min(consume_per_tick, len(queue.heap))):
            delivered.append(int(queue.poll().attributes["seq"]))
    return produced, delivered, len(queue.heap), stalled_ticks


produced, delivered, queued, stalled = simulate("oldest_first")
assert produced == len(delivered) + queued  # back pressure stalls the source; it never loses data
assert stalled > 0 and queued <= 200 + 50
assert delivered == sorted(delivered)       # FIFO-like: oldest leaves first
_, newest_delivered, _, _ = simulate("newest_first")
assert newest_delivered != sorted(newest_delivered)  # same throughput, different order
print(f"produced={produced} delivered={len(delivered)} queued={queued} producer stalled {stalled} ticks")

### Provenance

Provenance answers concrete questions: which processor modified this record, what the content looked like at that point, how long each hop took, and where a given input file ended up. It also enables replay: you can re-inject a FlowFile from a provenance event.

A provenance event records one transition: the component, the event type (`CREATE`, `RECEIVE`, `SEND`, `DROP`, `ROUTE`, `FORK`, `JOIN`, `ATTRIBUTES_MODIFIED`, `CONTENT_MODIFIED`), the FlowFile uuid, the attributes at that point, and the timing. The repository indexes configurable fields so you can search by `filename` or a custom business key.

Worked storage arithmetic. At 5,000 events/s with an average of three provenance events per FlowFile and a 200 byte event record, the repository writes about 3 MB/s. Retaining seven days needs 5,000 x 3 x 200 x 86,400 x 7 bytes, which is roughly 1.8 TB. That is why retention and indexed fields are capacity decisions, not defaults to ignore.

Common mistake: indexing every attribute. Each indexed field consumes index space and write time, and a wide index slows the repository without helping the queries you actually run. Index the few fields used for audits and debugging, such as `filename`, `path`, and one business key.

## Building flows

### Reliable flow design

A reliable flow uses back pressure on every connection, explicit failure and retry routing, idempotent sinks, and provenance. The retry path uses `RetryFlowFile` with a maximum count and penalization, so a failing FlowFile does not spin and a poison record eventually leaves the main path.

+ **Backpressure on every connection**, sized to the downstream system's true capacity.
+ **Retry with penalization** for transient failures: route to `RetryFlowFile` with a retry count and backoff instead of hammering the target.
+ **Dead-letter storage** keyed by reason, so operators can reprocess a known-bad batch.
+ **Idempotent sinks**: object keys derived from `filename` plus a content hash, and database upserts on a business key.
+ **Provenance enabled and retained** long enough for audit and replay.
+ **Parameter contexts** for environment-specific configuration, and versioned flows for promotion.
+ **Monitoring on queue depth and bulletin rate**, not only on processor errors.

Worked retry design:

```text
ConsumeKafka -> ValidateRecord --valid----> PutS3Object
                              --invalid--> PutFile (dead letter)
                              --failure--> RetryFlowFile (5 tries, 30 s penalty)
                                              --retries_exceeded--> PutFile (dead letter)
```

Every retry re-sends the same FlowFile, so `PutS3Object` must derive its object key from `filename` plus a content hash to stay idempotent.

Common mistake: a retry loop with no exit. Without `Maximum Retries` and a `retries_exceeded` route, a FlowFile that always fails occupies a queue forever, and the queue grows until back pressure stalls the whole flow. Always terminate retries into a visible dead-letter path.

In [ ]:
def put_s3(flowfile, attempt_outcomes):
    """A flaky destination: returns 'success' or 'failure' from a scripted sequence."""
    return attempt_outcomes.pop(0) if attempt_outcomes else "success"


def deliver(flowfile, outcomes, max_retries=5, penalty_s=30):
    """ValidateRecord -> PutS3Object with RetryFlowFile and a dead-letter path."""
    retries, waited = 0, 0
    while True:
        if put_s3(flowfile, outcomes) == "success":
            return "s3", retries, waited
        if retries == max_retries:
            return "dead_letter", retries, waited  # retries_exceeded: visible, not lost
        retries += 1
        waited += penalty_s  # penalized FlowFiles are not retried in a hot loop


transient = deliver(incoming[1], ["failure", "failure"])
poison = deliver(incoming[2], ["failure"] * 100)
assert transient == ("s3", 2, 60)
assert poison[0] == "dead_letter" and poison[1] == 5
print("transient failure ->", transient, "| poison record ->", poison)

### Expression Language and record processing

Expression language operates on attributes and is evaluated per FlowFile inside processor properties. Record processing uses a RecordReader to parse a stream into records and a RecordWriter to serialize them, so one processor handles JSON, Avro, CSV, or Parquet without per-format code. Both are controller services configured once and shared across processors.

```text
${filename}                                  -> orders_20261004.json
${filename:substringBefore('.')}             -> orders_20261004
${now():format('yyyy-MM-dd')}                -> 2026-10-04
${amount:toNumber():ge(10000)}               -> true / false
${status:equals('cancelled')}                -> true / false
${attr:replaceNull('unknown')}               -> default that avoids nulls
${path:append('/archive')}                   -> /landing/orders/archive
```

```text
ConsumeKafka (JSON) -> ConvertRecord (JsonTreeReader -> AvroRecordSetWriter)
                    -> QueryRecord (filter amount > 0)
                    -> PartitionRecord (by order_date)
                    -> PutS3Object (partitioned Parquet)
```

Worked record setup with a schema registry:

```json
{
  "controllerServices": [
    {"name": "JsonTreeReader", "type": "org.apache.nifi.json.JsonTreeReader",
     "properties": {"schema-access-strategy": "schema-name", "schema-registry": "acme-registry"}},
    {"name": "ParquetRecordSetWriter", "type": "org.apache.nifi.parquet.ParquetRecordSetWriter",
     "properties": {"schema-access-strategy": "inherit-record-schema"}}
  ]
}
```

`ValidateRecord` uses the reader to check every record against the registered schema and routes violations to `invalid` instead of writing bad rows.

Common mistake: using `infer-schema` in production. The reader guesses types from the first batch, so a field that is an integer in one file and a string in the next causes a schema conflict or a silent type change. Pin the schema to a registry subject so validation is deterministic.

### Custom code

NiFi is Java, and extensions are packaged as NAR files. `ExecuteScript` runs JSR-223 languages in-process, `InvokeScriptedProcessor` implements a processor interface in Groovy, and `ExecuteStreamCommand` shells out to any executable. All three run inside the NiFi JVM or as a child process, so a bad script degrades the whole node rather than one task.

+ `ExecuteScript`: runs Groovy, Jython, JRuby, or JavaScript (JSR-223) against a FlowFile in-process.
+ `InvokeScriptedProcessor`: defines a custom processor interface in Groovy with `onTrigger`.
+ `ExecuteStreamCommand`: shells out to any CLI, including Python, R, or a compiled binary.
+ Custom NAR: a packaged Java extension for production-grade processors with dependencies.

Common mistake: assuming the scripting language is the Python you use elsewhere. `ExecuteScript` Jython is Python 2 era, so f-strings and modern libraries are unavailable, and each invocation starts a script engine. For work on the critical path, use a native processor or a packaged Java extension and keep scripts for glue.

In [ ]:
flowfiles = 1_000_000
native_ms = 0.2
script_ms = 5.0

saved_seconds = flowfiles * (script_ms - native_ms) / 1000
print(f"1,000,000 FlowFiles at {script_ms} ms = {flowfiles * script_ms / 1000:,.0f} s of script time")
print(f"the same work at {native_ms} ms        = {flowfiles * native_ms / 1000:,.0f} s")
print(f"scripting overhead avoided             = {saved_seconds:,.0f} s")

assert round(saved_seconds) == 4800

## Configuration, promotion and security

### Parameter contexts versus the variable registry

The variable registry is a properties file loaded at startup and referenced with `${NAME}` syntax. It is global to the instance, read at startup rather than changed at runtime, and identical for every process group, which is why it is now the legacy mechanism.

Worked replacement. A parameter context bound to a process group defines `S3_BUCKET = acme-raw-prod` and `DB_PASSWORD` as a sensitive value. The flow references `#{S3_BUCKET}` and `#{DB_PASSWORD}`, and parameter context inheritance lets a child group reuse its parent's values.

Common mistake: using the variable registry for environment-specific values. There is one file per instance, so promoting the same flow to another environment either requires editing the file or silently ships the wrong value. Parameter contexts are per-environment objects that travel with the flow.

### Sensitive properties and templates

Sensitive properties are encrypted at rest with `nifi.sensitive.props.key` and are deliberately omitted from template XML. After an import the property is empty, so the processor is invalid until an operator re-enters the secret. That is the intended behavior rather than a bug.

Worked modern answer. Keep the secret in a parameter context marked sensitive and reference `#{db.password}` from the processor property. The versioned flow then contains a parameter reference instead of a value, and each environment supplies its own secret when the flow is applied.

Common mistake: sharing `nifi.sensitive.props.key` across environments or baking it into a container image. Anyone holding the key and a flow configuration can decrypt every sensitive property on every node. Keep it in a secret manager and treat rotation as a planned, coordinated operation.

A template is an XML snapshot of a process group. Export captures component configuration and wiring, but it strips sensitive property values and carries no version history. Import instantiates a fresh copy with new component ids, so a template cannot be updated in place or diffed between environments.

Worked migration problem. A template exported from dev contains `Bucket = acme-dev` as a plain property. Importing it into prod silently writes to the dev bucket until someone notices the data in the wrong place. A versioned flow with `#{bucket}` from a parameter context cannot make that mistake.

Common mistake: using templates for promotion. Sensitive values are removed, non-sensitive values leak across environments, and there is no version number to roll back to. Use a flow registry with parameter contexts for real promotion and keep templates for throwaway demos.

### Versioned flows and promotion

A versioned flow is a process group registered with a flow registry. Each commit creates a numbered version with an author and a comment, and the registry stores the flow JSON plus the referenced parameter contexts. Promotion applies a version to a target environment that supplies its own parameter values.

```text
Developer -> NiFi dev -> commit version -> NiFi Registry / Git
                                              |
                     UAT NiFi <- apply version + UAT parameter context
                                              |
                    Prod NiFi <- apply version + prod parameter context
```

Worked promotion sequence: dev commits version 12; UAT applies version 12 with the UAT parameter context; prod applies the same version with the prod context. A rollback is applying version 11, which is a registry operation rather than a rebuild from a template.

Common mistake: committing a process group before its parameter contexts are registered. The version then references parameters that do not exist in the target, and every environment reports invalid processors. Register the contexts first, then commit the flow that references them.

### Securing a deployment

Security has four layers: TLS for transport, authentication for identity, authorization for policy, and site-to-site trust for node-to-node and edge-to-core communication. TLS covers the UI, the REST API, the cluster protocol, and site-to-site transport, and NiFi can generate an internal keystore for a cluster.

```text
nifi.security.keystore=/opt/nifi/conf/keystore.jks
nifi.security.keystoreType=JKS
nifi.security.keystorePasswd=<encrypted>
nifi.security.keyPasswd=<encrypted>
nifi.security.truststore=/opt/nifi/conf/truststore.jks
nifi.security.truststoreType=JKS
nifi.security.truststorePasswd=<encrypted>
nifi.security.user.authorizer=managed-authorizer
nifi.security.user.login.identity.provider=ldap-provider
nifi.web.https.port=8443
nifi.remote.input.secure=true
nifi.cluster.protocol.is.secure=true
```

Worked layout. `nifi.security.user.authorizer = managed-authorizer` with policies per component and process group lets an operator read a flow without starting, stopping, or modifying it. Sensitive properties use `nifi.sensitive.props.key`, environment secrets live in parameter contexts, and access changes are recorded in the access log and in provenance.

Common mistake: granting broad write policy for convenience. A user who can modify any processor can redirect a destination, disable validation, or exfiltrate data. Least privilege means read access for most operators and write access only for the owners of a flow.

## Operations

### Monitoring, reporting tasks and bulletins

Monitoring answers two questions: is the flow keeping up, and where is the bottleneck. Queue depth and back pressure are the leading indicators; CPU, heap, GC pause, repository IOPS, and bulletin rate explain why. Tuning levers in rough order are `concurrentTasks`, run duration batching, record processors instead of regex, and repository or heap sizing.

A reporting task is a controller-level extension that runs on a schedule and reads NiFi's own metrics and state, never FlowFile content. It is created at the controller level rather than inside a process group, and its `schedulingStrategy` is TIMER_DRIVEN with a period such as `60 sec`.

Worked configuration. A `PrometheusReportingTask` with `Metrics Reporting Strategy = All Components`, `Send JVM Metrics = true`, and a 30 second period exposes queue depth, processor throughput, and heap as scrapeable metrics. Alerting on connection queue depth is what catches a stalled processor before back pressure spreads upstream.

Common mistake: relying on the UI alone for history. Bulletins live in an in-memory ring buffer and the indicator only shows for a few minutes, so a failure at 02:00 is invisible by morning. A reporting task that ships metrics to Prometheus or another store is what makes an overnight failure reviewable.

A bulletin is an in-memory notice created when a component logs at its bulletin level or above, or when the framework reports a problem with a component. Bulletins live in a `VolatileBulletinRepository` with a bounded ring buffer, so older entries are displaced and nothing survives a node restart.

Worked noise problem. A processor that retries a refused connection once per second emits up to 3,600 bulletins an hour, fills the ring buffer, and hides every other component's messages. Fixing the missing route removes the source of the noise; suppressing the display only hides the symptom.

Common mistake: raising a processor's bulletin log level from WARN to ERROR to reduce noise. That hides warnings such as pool exhaustion or slow responses, which are the early signal, so the first visible symptom becomes a full outage instead of a degraded flow.

### Tuning a flow

Tuning a NiFi flow is a three-step loop: find the connection that is filling, find the processor that is slow on that path, and change one setting. NiFi exposes both signals directly, so guessing is unnecessary. Queue depth is the symptom and processor time per FlowFile is the cause.

Back pressure is a throttle, not a target. Set the object and data-size thresholds to what the downstream can absorb and the node can hold, not to the largest number that silences warnings. A useful starting point is the downstream's capacity times its recovery time: an API that accepts 500 files per second and recovers in 60 seconds suggests a threshold near 30,000 objects.

`concurrentTasks` is parallelism per processor and run duration is how long a task holds its thread before yielding. For I/O-bound processors raise concurrency; for high-volume processors set run duration to one or two seconds so a single invocation processes many FlowFiles instead of one. `TIMER_DRIVEN` with a period suits polling sources, while `EVENT_DRIVEN` suits processors fed by a queue.

Worked throughput. A `ConvertRecord` taking 25 ms per FlowFile with `concurrentTasks = 4` processes about 160 FlowFiles/s. Raise concurrency to 16 and the same processor handles about 640 FlowFiles/s, provided the reader and writer services are thread-safe and the downstream connection does not immediately hit back pressure. If it does, this processor was never the bottleneck.

```json
{
  "component": {
    "name": "ConvertRecord",
    "config": {
      "schedulingStrategy": "TIMER_DRIVEN",
      "schedulingPeriod": "0 sec",
      "concurrentTasks": 16,
      "runDurationMillis": 1500,
      "properties": {
        "record-reader": "JsonTreeReader",
        "record-writer": "AvroRecordSetWriter"
      },
      "autoTerminatedRelationships": []
    }
  }
}
```

Common mistake: raising `concurrentTasks` on every processor at once. Throughput then looks unchanged because the queue moved to the slowest downstream stage, and the extra threads only add contention on shared controller services. Change one processor at a time and watch the connection that was filling.

### Sizing repositories and heap

Heap and disk do different jobs. Heap holds attributes, queue metadata, processor state, and repository indexes; disk holds content, the FlowFile repository, provenance, and the database repository. Sizing either one too small turns a smooth flow into garbage collection pauses or repository failures.

Attributes and queue metadata live in heap, so the heap must cover the number of queued FlowFiles times the average attribute bytes, plus processor working memory and JVM overhead. Two million queued FlowFiles with twenty attributes of 50 bytes each is about 2 GB of attribute data before content, which is why a very deep queue is a heap risk even when content is on disk.

Content sizing is rate x size x retention x safety factor. At 8,000 FlowFiles/s of 3 KB retained for ten minutes with a 2x factor, that is 24.6 MB/s x 600 s x 2, about 29.5 GB. The archive reserve (`nifi.content.repository.archive.max.usage.percentage`, default 50 percent) means the disk must be larger than that number, and it also holds provenance and the FlowFile repository.

Common mistake: sizing the content repository from the average FlowFile and forgetting the archive reserve and the provenance repository that share the same disk. A node then fails with a full disk while the computed content size still looks comfortable. Track free space per repository directory, not as one total.

### Clustering and scaling

There is no master from NiFi 1.0 onward. Every node runs the same software, ZooKeeper elects a cluster coordinator, and a separate election chooses a primary node. The coordinator manages membership and replicates flow configuration, while the primary node runs processors marked primary-node-only, such as `ListSFTP`.

Worked five-node cluster. With `ListSFTP` marked primary-node-only, one node lists the remote directory and distributes the FlowFiles, while all five run the fetch and write processors. Site-to-site moves data between a MiNiFi edge agent and the cluster using the same protocol, and it is also how one NiFi instance sends to another.

Common mistake: assuming the coordinator also processes all the data. The coordinator is an operational role, not a data supernode, so paying for a bigger coordinator node is wasted. Do not assume ZooKeeper is still required either: newer versions removed the external ensemble while keeping both roles.

By default a connection keeps FlowFiles on the node that created them, so one node that received a large file stays busy while others idle. Load balancing on a connection redistributes queued FlowFiles across the cluster at the cost of moving content over the network, and the strategies are `Do not load balance`, `Partition by attribute`, `Round robin`, and `Single node`.

Worked node math. A flow needs 12,000 FlowFiles/s and one node sustains about 2,000 FlowFiles/s for this processor mix. At a 70 percent target utilization the usable rate per node is 1,400 FlowFiles/s, so the cluster needs 12000 / 1400 = 8.6, which rounds to 9 nodes. The extra node also covers a single node failure without exceeding capacity.

Common mistake: scaling nodes to fix a bottleneck in a shared external system. If the destination database accepts 500 writes per second in total, ten NiFi nodes will not raise that ceiling; they will queue, retry, and fail faster. Scale or batch the destination before adding nodes.

## When not to use NiFi

NiFi is the wrong tool when the work is heavy compute, when the rate is millions of messages per second, or when a single piece of business logic belongs in a service. It is strongest at protocol-diverse ingress, per-record routing and enrichment, and auditable last-mile delivery.

+ **Complex stateful analytics**: joins across large datasets, windowed aggregations, and graph or ML workloads belong in Spark, Flink, or a warehouse. NiFi can trigger and route them.
+ **Sub-second, very high-volume streaming**: at millions of messages per second, Kafka plus a stream processor is cheaper per event; NiFi is better for ingress, enrichment, and delivery.
+ **Heavy business logic in code**: a flow with hundreds of `ExecuteScript` processors is an unmaintainable service written in the wrong language.
+ **Scheduled batch orchestration**: dependency-aware scheduling, backfills, and SLAs are Airflow's job; NiFi's scheduler is per-processor, not a DAG scheduler for external jobs.
+ **Simple, single-step pipelines**: a cron job or a managed connector is less operational overhead than a NiFi cluster.
+ **Teams without operational capacity**: a production NiFi cluster means TLS, ZooKeeper or equivalent coordination, disk sizing, heap tuning, and upgrade management.
+ **Exactly-once semantics everywhere**: NiFi provides reliable delivery with retries, not exactly-once end to end; you still need idempotent sinks.

Worked comparison with Flume and Kafka Connect. For SFTP and MQTT ingress with per-record validation and lineage, NiFi wins. For a durable replayable backbone, Kafka wins. For declarative connectors whose offsets live in Kafka, Kafka Connect wins. Flume wins only for Hadoop-native HDFS or HBase delivery with an edge buffer.

Common mistake: choosing a tool by ecosystem rather than by contract. If the requirement is replay with many independent consumers, no amount of NiFi routing substitutes for Kafka retention. If the requirement is per-record enrichment with an audit trail, Kafka alone does not provide it.

## Real world use cases

### 1. E-commerce order events into a lakehouse

Business context: an e-commerce platform must land order events into object storage within minutes and reject malformed events visibly. Volume: ~40M events/day, peak 8k events/s, ~3 KB per event.

```text
ConsumeKafka (orders.events) -> ValidateRecord (schema from Schema Registry)
      --valid--> UpdateRecord (add ingest_date, source) -> PutS3Object (Parquet, dt=)
      --invalid--> PutFile (/dead-letter/orders/{reason}/)
```

```json
{
  "component": {
    "name": "PutS3Object",
    "type": "org.apache.nifi.processors.aws.s3.PutS3Object",
    "config": {
      "properties": {
        "Bucket": "acme-raw-prod",
        "Object Key": "orders/dt=${ingest_date}/${filename}",
        "Content Type": "application/octet-stream",
        "Server Side Encryption": "AES256"
      }
    }
  }
}
```

Why this design: NiFi absorbs protocol and schema churn at the edge, routes invalid records to an inspectable dead-letter tree instead of dropping them, and writes date-partitioned Parquet that Spark and the warehouse can read directly. Backpressure protects S3 from request storms.

### 2. Fintech regulatory file ingestion

Business context: a payments firm ingests daily settlement and AML files from banks over SFTP and must prove chain of custody. Volume: ~2,000 files/day, 50 MB-5 GB each, strict audit.

```text
ListSFTP -> FetchSFTP -> ValidateRecord -> RouteOnAttribute --valid--> PutHDFS/PutS3
                                                          --invalid--> PutFile (quarantine)
Provenance retained 90 days; every file tied to a bank, batch id, and checksum
```

```text
nifi.provenance.repository.max.storage.time=90 days
nifi.provenance.repository.indexed.fields=filename,path,sourceSystem,batchId
nifi.content.repository.archive.max.usage.percentage=70%
```

Why this design: NiFi's provenance gives the audit trail regulators ask for, primary-node-only listing avoids duplicate pickup across the cluster, validation happens before data reaches the warehouse, and quarantine keeps failures reviewable rather than silently retried into the lake.

### 3. Partner file drops into a multi-tenant SaaS lake

Business context: a B2B SaaS platform onboards 300 partners who upload CSV and JSON files to per-tenant SFTP directories; data must be isolated, validated, and attributed. Volume: ~15 TB/month, highly uneven per tenant.

```text
ListSFTP (primary node) -> FetchSFTP -> ExtractText (tenant from path)
                        -> ValidateRecord (tenant schema)
                        --valid--> PutS3Object (s3://acme-lake/tenant=${tenant}/...)
                        --invalid--> PutFile (/quarantine/${tenant}/)
```

```text
${path:substringAfter('/upload/'):substringBefore('/')}   -> tenant id
${filename:toLower():endsWith('.csv')}                    -> format routing
```

Why this design: tenant is extracted from the path with expression language, so no per-tenant flow is needed; isolated S3 prefixes and quarantine directories make data residency and blast radius explicit; and backpressure per connection stops one enormous tenant upload from starving the others.

## NiFi exercises

The units match what NiFi reports: FlowFiles per second for rate, bytes for content and provenance, and seconds for recovery windows. Treat the asserted numbers as a check on the formula rather than a prediction of a real node, because production adds disk variance, garbage collection, and retries. The value is knowing which quantity decides the setting.

### Try it 1. Compute back pressure thresholds from queue growth

**Task.** A connection receives 400 FlowFiles/s and 1.5 MiB/s of content. You want a 120 second recovery window before the upstream processor is throttled. Compute the object threshold and the data-size threshold in MiB.

**Hint.** A threshold is the growth rate times the recovery window: `objects = rate x seconds` and `size = bytes_per_s x seconds`.

**Expected result.** An object threshold of `48,000` FlowFiles and a data-size threshold of `180.0` MiB.

In [ ]:
object_growth_per_s = 400
size_growth_mib_per_s = 1.5
recovery_seconds = 120

object_threshold = object_growth_per_s * recovery_seconds
size_threshold_mib = size_growth_mib_per_s * recovery_seconds

print(f"object threshold = {object_threshold:,} FlowFiles")
print(f"size threshold   = {size_threshold_mib:.1f} MiB")

assert object_threshold == 48_000
assert size_threshold_mib == 180.0

### Try it 2. Size the content repository

**Task.** A flow ingests 8,000 FlowFiles/s at 3 KiB each and the slowest branch retains ten minutes of data. With a 2x safety factor for claim overhead and archive cleanup, compute the content size in GiB and the raw disk needed if the archive reserve is 50 percent.

**Hint.** `bytes = rate x size x seconds x safety`; `GiB = bytes / 1024**3`; divide by `1 - archive_pct` for the disk.

**Expected result.** About `27.47` GiB of content and `54.93` GiB of disk.

In [ ]:
rate_per_s = 8_000
avg_kib = 3
retention_seconds = 600
safety_factor = 2
archive_pct = 0.5

content_bytes = rate_per_s * avg_kib * 1024 * retention_seconds * safety_factor
content_gib = content_bytes / 1024 ** 3
disk_gib = content_gib / (1 - archive_pct)

print(f"content = {content_bytes:,} bytes = {content_gib:.2f} GiB")
print(f"disk    = {disk_gib:.2f} GiB")

assert round(content_gib, 2) == 27.47
assert round(disk_gib, 2) == 54.93

### Try it 3. Choose a run schedule from throughput

**Task.** A processor has a 20 ms fixed cost per invocation and 0.5 ms per FlowFile. Compare one FlowFile per invocation with a 2000 ms run duration that batches work, then compute the speedup from batching.

**Hint.** With run duration 0 every invocation handles one FlowFile. With 2000 ms the invocation handles `run_duration / per_flowfile_ms` FlowFiles and costs the fixed overhead plus the run duration.

**Expected result.** About `48.78` FlowFiles/s unbatched, about `1,980.20` FlowFiles/s batched, a speedup of about `40.59x`.

In [ ]:
overhead_ms = 20.0
per_flowfile_ms = 0.5
run_duration_ms = 2000.0

ff_per_s_unbatched = 1000.0 / (overhead_ms + per_flowfile_ms)
flowfiles_per_invocation = run_duration_ms / per_flowfile_ms
ff_per_s_batched = flowfiles_per_invocation * 1000.0 / (overhead_ms + run_duration_ms)
speedup = ff_per_s_batched / ff_per_s_unbatched

print(f"unbatched = {ff_per_s_unbatched:,.2f} FlowFiles/s")
print(f"batched   = {ff_per_s_batched:,.2f} FlowFiles/s")
print(f"speedup   = {speedup:.2f}x")

assert round(ff_per_s_unbatched, 2) == 48.78
assert round(ff_per_s_batched, 2) == 1980.20
assert round(speedup, 2) == 40.59

### Try it 4. Plan a batch size for a record writer

**Task.** A Parquet writer should roll a file at about 128 MiB and records average 2 KiB. Compute the records per batch and how long the batch takes to fill at 50,000 records/s.

**Hint.** `records = target_mib x 1024**2 // record_bytes`; then divide by the write rate.

**Expected result.** `65,536` records per batch, filling in about `1.31` seconds.

In [ ]:
target_mib = 128
record_bytes = 2048
write_rate_per_s = 50_000

records_per_batch = target_mib * 1024 ** 2 // record_bytes
seconds_to_fill = records_per_batch / write_rate_per_s

print(f"records per batch = {records_per_batch:,}")
print(f"seconds to fill   = {seconds_to_fill:.2f}")

assert records_per_batch == 65_536
assert round(seconds_to_fill, 2) == 1.31

### Try it 5. Estimate cluster nodes from throughput

**Task.** A flow must handle 12,000 FlowFiles/s. One node sustains about 2,000 FlowFiles/s for this processor mix, and you want a 70 percent target utilization. Compute the node count and confirm that the cluster still meets demand after one node fails.

**Hint.** `usable_per_node = per_node x utilization`; round up `demand / usable_per_node`. Then check `(nodes - 1) x per_node` against demand.

**Expected result.** `9` nodes, and eight healthy nodes still deliver `16,000` FlowFiles/s, above demand.

In [ ]:
import math

demand_per_s = 12_000
per_node_per_s = 2_000
target_utilization = 0.7

usable_per_node = per_node_per_s * target_utilization
nodes = math.ceil(demand_per_s / usable_per_node)
capacity_after_failure = (nodes - 1) * per_node_per_s

print(f"usable per node          = {usable_per_node:,.0f} FlowFiles/s")
print(f"nodes required           = {nodes}")
print(f"capacity after 1 failure = {capacity_after_failure:,} FlowFiles/s")

assert nodes == 9
assert capacity_after_failure >= demand_per_s

### Try it 6. Compute provenance storage

**Task.** A flow produces 5,000 FlowFiles/s, records three provenance events per FlowFile on average, and each event record is about 200 bytes. You retain provenance for seven days. Compute the total storage in TiB.

**Hint.** `bytes = rate x events x event_bytes x 86400 x days`; convert with `1024**4` for TiB.

**Expected result.** About `1.65` TiB of provenance for the seven day window.

In [ ]:
rate_per_s = 5_000
events_per_flowfile = 3
event_bytes = 200
retention_days = 7

total_bytes = rate_per_s * events_per_flowfile * event_bytes * 86_400 * retention_days
total_tib = total_bytes / 1024 ** 4

print(f"events/s = {rate_per_s * events_per_flowfile:,}")
print(f"total    = {total_bytes:,} bytes = {total_tib:.2f} TiB")

assert round(total_tib, 2) == 1.65

### Try it 7. Size concurrent tasks from latency

**Task.** A processor takes 40 ms per FlowFile and must sustain 800 FlowFiles/s on one node. Compute the concurrent tasks needed, using the latency-throughput relationship.

**Hint.** `tasks = ceil(target_rate x latency_seconds)`, because each task can complete one FlowFile per latency window.

**Expected result.** `32` concurrent tasks.

In [ ]:
import math

target_per_s = 800
per_flowfile_ms = 40

latency_seconds = per_flowfile_ms / 1000
tasks = math.ceil(target_per_s * latency_seconds)

print(f"target rate = {target_per_s:,} FlowFiles/s")
print(f"latency     = {per_flowfile_ms} ms")
print(f"tasks       = {tasks}")

assert tasks == 32

### Try it 8. Estimate compression savings for records

**Task.** A flow writes 4 TB of raw JSON per day. Converting to Parquet reduces the volume to 30 percent of raw, and enabling Snappy compression on the writer halves it again. Compute the final daily size and the storage saved.

**Hint.** Multiply the raw volume by each factor in turn, then subtract from the raw volume.

**Expected result.** `0.6` TB per day written, `3.4` TB per day saved.

In [ ]:
raw_tb_per_day = 4.0
parquet_factor = 0.3
compression_factor = 0.5

final_tb_per_day = raw_tb_per_day * parquet_factor * compression_factor
saved_tb_per_day = raw_tb_per_day - final_tb_per_day

print(f"final size = {final_tb_per_day:.2f} TB/day")
print(f"saved      = {saved_tb_per_day:.2f} TB/day")

assert round(final_tb_per_day, 2) == 0.6
assert round(saved_tb_per_day, 2) == 3.4